# Lesson 7 — First Quantum Algorithms: Deutsch-Jozsa and Bernstein-Vazirani

**Quantum Computing with Qiskit**

This notebook accompanies Lesson 7. Work through the cells in order. Every cell is meant to be run; modify the code freely and re-run to experiment.

**Topics covered:**
- The oracle model and phase kickback
- Deutsch-Jozsa algorithm: constant vs balanced in one query
- Bernstein-Vazirani algorithm: recovering a hidden string in one query
- The prepare-query-interfere pattern

In [ ]:
!pip install qiskit qiskit-aer qiskit-ibm-runtime pylatexenc --quiet

In [ ]:
import qiskit
import qiskit_aer
import qiskit_ibm_runtime

print("Qiskit:             ", qiskit.__version__)
print("Qiskit Aer:         ", qiskit_aer.__version__)
print("Qiskit IBM Runtime: ", qiskit_ibm_runtime.__version__)

## 1. The Oracle Model and Phase Kickback

An oracle computes $f: \{0,1\}^n \to \{0,1\}$ as a black box. The quantum oracle (bit oracle) acts as:

$$U_f|x\rangle|y\rangle = |x\rangle|y \oplus f(x)\rangle$$

Setting the ancilla to $|{-}\rangle = (|0\rangle - |1\rangle)/\sqrt{2}$ causes the result to appear as a phase instead:

$$U_f|x\rangle|{-}\rangle = (-1)^{f(x)}|x\rangle|{-}\rangle$$

This is **phase kickback**. The ancilla is unchanged; the phase $(-1)^{f(x)}$ is kicked back to the input register.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
import numpy as np

sim = AerSimulator()

# Demonstrate phase kickback for f(x) = x (identity on 1 qubit)
# Oracle: CNOT from q0 to ancilla q1
qc_pk = QuantumCircuit(2)
qc_pk.x(1)           # ancilla |1>
qc_pk.h([0, 1])      # q0 -> |+>, q1 -> |->
qc_pk.barrier()
qc_pk.cx(0, 1)       # oracle: f(x) = x

sv = Statevector.from_label('00').evolve(qc_pk)
print("Statevector after phase kickback (q0 | ancilla):")
print(sv.data.round(4))
print()
print("Amplitudes at |00>, |01>, |10>, |11>:")
for i, amp in enumerate(sv.data):
    label = format(i, '02b')
    print(f"  |{label}> = {amp.real:+.4f}")
print()
print("Phase at |0>_q0: +, at |1>_q0: -  (phase kicked into input qubit)")

The ancilla qubit stays in $|{-}\rangle$. The input qubit picks up a relative phase:
- $f(0)=0$: no phase flip on $|0\rangle$
- $f(1)=1$: phase flip on $|1\rangle$

After kickback, the input register holds $(|0\rangle - |1\rangle)/\sqrt{2} = |{-}\rangle$, and the ancilla still holds $|{-}\rangle$.

## 2. Deutsch-Jozsa

**Problem:** Given oracle for $f: \{0,1\}^n \to \{0,1\}$ guaranteed to be constant or balanced, which is it?

**Algorithm:**
1. Prepare $|0\rangle^{\otimes n}|1\rangle$
2. $H^{\otimes(n+1)}$: input $\to \frac{1}{\sqrt{2^n}}\sum_x|x\rangle$, ancilla $\to |{-}\rangle$
3. Oracle: $\frac{1}{\sqrt{2^n}}\sum_x(-1)^{f(x)}|x\rangle|{-}\rangle$
4. $H^{\otimes n}$ on input
5. Measure: $0^n$ means constant, any nonzero means balanced

**Why:** Amplitude at $|0^n\rangle$ after step 4 is $\frac{1}{2^n}\sum_x(-1)^{f(x)}$, which equals $\pm 1$ for constant $f$ and $0$ for balanced $f$.

In [ ]:
# Oracle constructors

def dj_constant_oracle(n, value=0):
    """Constant oracle: f(x) = value for all x."""
    oracle = QuantumCircuit(n + 1, name='Oracle')
    if value:
        oracle.x(n)   # flip ancilla: Uf(|y>) = |y XOR 1>
    return oracle

def dj_balanced_oracle(n, target_qubits=None):
    """
    Balanced oracle: f(x) = XOR of bits at target_qubits.
    Defaults to qubit 0, giving f(x) = x_0.
    """
    if target_qubits is None:
        target_qubits = [0]
    oracle = QuantumCircuit(n + 1, name='Oracle')
    for q in target_qubits:
        oracle.cx(q, n)   # CNOT from input qubit q to ancilla
    return oracle

# Deutsch-Jozsa circuit
def deutsch_jozsa(oracle, n):
    """
    Deutsch-Jozsa circuit for n input qubits.
    oracle: QuantumCircuit on n+1 qubits (inputs 0..n-1, ancilla n).
    """
    qc = QuantumCircuit(n + 1, n)
    qc.x(n)              # ancilla |1>
    qc.barrier()
    qc.h(range(n + 1))   # all qubits: |+>^n |->  
    qc.barrier()
    qc.compose(oracle, inplace=True)
    qc.barrier()
    qc.h(range(n))       # inverse Hadamard on input register
    qc.barrier()
    qc.measure(range(n), range(n))
    return qc

# Draw the circuit
n = 3
oracle_example = dj_balanced_oracle(n, target_qubits=[0, 1, 2])
qc_example = deutsch_jozsa(oracle_example, n)
qc_example.draw('mpl')

In [ ]:
# Run all four oracle types
n = 3
all_zeros = '0' * n

test_cases = [
    ('constant f=0',         dj_constant_oracle(n, value=0)),
    ('constant f=1',         dj_constant_oracle(n, value=1)),
    ('balanced f=x₀',        dj_balanced_oracle(n, target_qubits=[0])),
    ('balanced f=x₀⊕x₁⊕x₂', dj_balanced_oracle(n, target_qubits=[0, 1, 2])),
]

print(f"Deutsch-Jozsa results (n={n}):")
print()
for label, oracle in test_cases:
    qc = deutsch_jozsa(oracle, n)
    counts = sim.run(qc, shots=1024).result().get_counts()
    result = max(counts, key=counts.get)
    verdict = 'CONSTANT' if result == all_zeros else 'BALANCED'
    print(f"  {label:30s}: measured {result!r}  -> {verdict}")

Both constant oracles produce `'000'` with certainty. Both balanced oracles produce a nonzero result.

**Classical query comparison.** For $n=3$, the classical worst case is $2^{n-1}+1 = 5$ queries. The quantum algorithm always uses 1.

In [ ]:
# Verify via statevector: show amplitude at |000> for each oracle type
print("Statevector analysis: amplitude at |000> after DJ circuit (no measurement)")
print()

for label, oracle in test_cases:
    # Build circuit without measurement
    qc_sv = QuantumCircuit(n + 1)
    qc_sv.x(n)
    qc_sv.h(range(n + 1))
    qc_sv.compose(oracle, inplace=True)
    qc_sv.h(range(n))

    sv = Statevector.from_label('0' * (n + 1)).evolve(qc_sv)
    # Index 0 of statevector corresponds to |000...0> on input qubits (ancilla in any state)
    # Sum over ancilla: amplitude at |000> on input = sum of sv[0] and sv[1] (ancilla=0,1)
    amp_at_zero = sum(abs(sv.data[i])**2 for i in range(2) if i < len(sv.data))
    # More directly: check P(000x) for ancilla x=0 or x=1
    probs = sv.probabilities_dict()
    p_zero_input = sum(v for k, v in probs.items() if k[1:] == '0' * n)
    print(f"  {label:30s}: P(input=000) = {p_zero_input:.4f}")

## 3. Bernstein-Vazirani

**Problem:** Given oracle for $f(x) = s \cdot x \bmod 2$ where $s \in \{0,1\}^n$ is unknown, find $s$.

**Classical requirement:** $n$ queries — one per bit, querying $f(e_i)$ gives $s_i$.

**Quantum:** 1 query. The circuit is identical to Deutsch-Jozsa.

**Why:** The phases $(-1)^{s \cdot x}$ are exactly the Hadamard transform of $|s\rangle$:
$$H^{\otimes n}|s\rangle = \frac{1}{\sqrt{2^n}}\sum_x (-1)^{s \cdot x}|x\rangle$$

Applying $H^{\otimes n}$ again returns $|s\rangle$. The measurement outcome is $s$ with certainty.

The amplitude at $|z\rangle$ after the final $H^{\otimes n}$ is:
$$a_z = \frac{1}{2^n}\sum_x (-1)^{x \cdot (s \oplus z)} = \begin{cases} 1 & z = s \\ 0 & z \ne s \end{cases}$$

In [ ]:
def bv_oracle(secret):
    """
    Oracle for f(x) = s·x mod 2.
    secret: binary string in Qiskit bitstring order
            (secret[0] is the MSB, corresponds to the highest qubit index).
    The measured bitstring equals secret directly.
    """
    n = len(secret)
    oracle = QuantumCircuit(n + 1, name='BV Oracle')
    for i, bit in enumerate(reversed(secret)):
        # reversed: secret[n-1-i] maps to qubit i
        if bit == '1':
            oracle.cx(i, n)
    return oracle

def bernstein_vazirani(oracle, n):
    return deutsch_jozsa(oracle, n)   # exact same circuit

# Draw the oracle for secret='101'
oracle_bv = bv_oracle('101')
oracle_bv.draw('mpl')

In [ ]:
# Test recovery for several secrets
secrets = ['101', '1011', '10110100', '0110101011001101']

print("Bernstein-Vazirani recovery:")
print()
for s in secrets:
    n = len(s)
    oracle = bv_oracle(s)
    qc = bernstein_vazirani(oracle, n)
    counts = sim.run(qc, shots=1024).result().get_counts()
    measured = max(counts, key=counts.get)
    print(f"  n={n:2d}  secret = {s!r:20s}  measured = {measured!r:20s}  match = {measured == s}")

In [ ]:
# Draw the full BV circuit for secret='101'
qc_bv = bernstein_vazirani(bv_oracle('101'), 3)
qc_bv.draw('mpl')

Every secret string is recovered in a single query, regardless of length. For the 16-bit case, a classical algorithm would require 16 queries; the quantum algorithm still used 1.

## 4. Classical vs Quantum Query Count

To make the advantage concrete, simulate a classical BV solver and count its queries.

In [ ]:
def make_classical_bv_oracle(secret):
    """Return a classical function computing f(x) = s·x mod 2."""
    def f(x_int, n):
        # secret[0] is MSB (qubit n-1), so int(secret, 2) places qubit i at bit i directly
        s_int = int(secret, 2)
        return bin(x_int & s_int).count('1') % 2
    return f

def classical_bv(secret):
    """
    Recover secret by querying f at each basis vector e_i.
    Returns (recovered_secret, number_of_queries).
    """
    n = len(secret)
    f = make_classical_bv_oracle(secret)
    recovered_bits = []
    queries = 0
    for i in range(n):
        e_i = 1 << i          # basis vector: only bit i is 1
        s_i = f(e_i, n)
        recovered_bits.append(str(s_i))
        queries += 1
    # recovered_bits[i] = s_i (qubit i = LSB). Reverse for MSB-first output.
    recovered = ''.join(reversed(recovered_bits))
    return recovered, queries

print("Classical vs Quantum query count for Bernstein-Vazirani:")
print()
print(f"  {'secret':20s}  {'classical queries':>18s}  {'quantum queries':>16s}")
print(f"  {'-'*20}  {'-'*18}  {'-'*16}")

for s in secrets:
    recovered, q_classical = classical_bv(s)
    q_quantum = 1
    assert recovered == s, f"Classical mismatch: {recovered} != {s}"
    print(f"  {s:20s}  {q_classical:>18d}  {q_quantum:>16d}")

The quantum algorithm always uses 1 query. The classical algorithm scales linearly with $n$. For $n=1000$, that is a 1000x reduction in oracle calls.

## 5. Interference Visualised

To see the interference at work, inspect the statevector just before the final measurement for both a constant and a balanced oracle.

In [ ]:
import matplotlib.pyplot as plt

n = 2
oracles_to_plot = [
    ('constant f=0',   dj_constant_oracle(n, 0)),
    ('balanced f=x₀',  dj_balanced_oracle(n, [0])),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (label, oracle) in zip(axes, oracles_to_plot):
    # Circuit without measurement, without final H
    qc_pre = QuantumCircuit(n + 1)
    qc_pre.x(n); qc_pre.h(range(n + 1))
    qc_pre.compose(oracle, inplace=True)
    sv_pre = Statevector.from_label('0' * (n + 1)).evolve(qc_pre)

    # Circuit without measurement, WITH final H
    qc_post = qc_pre.copy()
    qc_post.h(range(n))
    sv_post = Statevector.from_label('0' * (n + 1)).evolve(qc_post)

    # Plot probabilities over input register (ignore ancilla)
    probs_pre  = [sum(abs(sv_pre.data[i + j * 2**n])**2 for j in range(2)) for i in range(2**n)]
    probs_post = [sum(abs(sv_post.data[i + j * 2**n])**2 for j in range(2)) for i in range(2**n)]

    x = np.arange(2**n)
    width = 0.35
    labels_x = [format(i, f'0{n}b') for i in range(2**n)]
    ax.bar(x - width/2, probs_pre,  width, label='After oracle (before 2nd H)', alpha=0.7)
    ax.bar(x + width/2, probs_post, width, label='After 2nd H (before measure)', alpha=0.7)
    ax.set_xticks(x); ax.set_xticklabels(labels_x)
    ax.set_ylim(0, 1.1)
    ax.set_xlabel('Input register state')
    ax.set_ylabel('Probability')
    ax.set_title(label)
    ax.legend(fontsize=8)

plt.suptitle('Interference before and after the second Hadamard')
plt.tight_layout()
plt.show()

For the constant oracle, the probabilities are spread evenly after the oracle but collapse entirely to $|00\rangle$ after the second Hadamard: constructive interference at $|0^n\rangle$.

For the balanced oracle, the probabilities are also spread evenly after the oracle, but the second Hadamard produces destructive interference at $|00\rangle$ and constructive interference at the state encoding the hidden structure.

## 6. Exercises

### Exercise 1: Verifying the decision rule

Run Deutsch-Jozsa for $n = 5$ with the following oracles and confirm the decision:
1. Constant oracle with $f = 0$
2. Constant oracle with $f = 1$
3. Balanced oracle $f(x) = x_2$ (only qubit 2 contributes)
4. Balanced oracle $f(x) = x_0 \oplus x_2 \oplus x_4$

For the balanced cases, state what nonzero bitstring you expect and verify it matches the measurement.

In [ ]:
# Exercise 1: your solution here
n = 5
all_zeros = '0' * n

# TODO: build four oracles and run DJ for each
# TODO: check verdict and explain the expected nonzero bitstrings for balanced cases

### Exercise 2: Classical Deutsch-Jozsa query counter

Implement a classical Deutsch-Jozsa solver that queries $f$ until it can decide.

1. Query $f(0)$ first. If any subsequent query returns a different value, conclude balanced immediately.
2. If after $2^{n-1}+1$ queries all values agree, conclude constant.
3. Run your classical solver for $n = 1, 2, 3, 4$ with a constant oracle and record the number of queries used.
4. What is the worst-case query count as a function of $n$? Compare it with the quantum algorithm.

In [ ]:
# Exercise 2: your solution here

def make_classical_dj_oracle(oracle_type, n):
    """
    Returns a classical function f(x_int) -> 0 or 1.
    oracle_type: 'constant0', 'constant1', or 'balanced'
    """
    if oracle_type == 'constant0':
        return lambda x: 0
    elif oracle_type == 'constant1':
        return lambda x: 1
    else:  # balanced: f(x) = bit 0 of x
        return lambda x: x & 1

def classical_dj(f, n):
    """Return ('constant' or 'balanced', number_of_queries)."""
    queries = 0
    # TODO: implement
    pass

# TODO: run for n=1..4 and report query counts

### Exercise 3: Bernstein-Vazirani with noise

Run the BV algorithm on a noisy simulator and see how noise affects the result.

1. Use `AerSimulator` with a depolarizing noise model: apply `depolarizing_error(p, 1)` on every single-qubit gate and `depolarizing_error(p, 2)` on every two-qubit gate.
2. Choose `secret = '10110100'` ($n=8$) and run with error rates $p \in \{0.001, 0.01, 0.05\}$.
3. For each noise level, report the most frequently measured bitstring and its fraction of shots.
4. At what noise level does the algorithm start returning the wrong answer?

In [ ]:
# Exercise 3: your solution here
from qiskit_aer.noise import NoiseModel, depolarizing_error

secret = '10110100'
n = len(secret)

def run_bv_noisy(secret, p, shots=4096):
    noise_model = NoiseModel()
    noise_model.add_all_qubit_quantum_error(depolarizing_error(p, 1), ['h', 'x'])
    noise_model.add_all_qubit_quantum_error(depolarizing_error(p, 2), ['cx'])

    n = len(secret)
    oracle = bv_oracle(secret)
    qc = bernstein_vazirani(oracle, n)
    sim_noisy = AerSimulator(noise_model=noise_model)
    counts = sim_noisy.run(qc, shots=shots).result().get_counts()
    top = max(counts, key=counts.get)
    return top, counts[top] / shots

for p in [0.001, 0.01, 0.05]:
    top, frac = run_bv_noisy(secret, p)
    correct = top == secret
    print(f"  p={p:.3f}: top answer = {top!r}  ({frac:.3f} of shots)  correct={correct}")

### Exercise 4: A balanced oracle that is not linear

Construct a balanced oracle for $n=3$ that is **not** of the form $f(x) = s \cdot x$ for any $s$.

One way: use X gates on some input qubits before and after the CNOT chain, which shifts the output pattern.
For example, apply X on qubit 0, CNOT(q0, ancilla), then X on qubit 0 again. This computes $f(x) = 1 - x_0$, which is still a linear-affine function. To get a truly non-linear balanced function, XOR two inputs: $f(x) = x_0 \cdot x_1$ (and balance manually).

1. Build a balanced oracle for $f(x) = x_0 \oplus x_1$ (balanced for $n=3$).
2. Run DJ and confirm the verdict is balanced.
3. The measured nonzero bitstring is the hidden $s$ for this linear function. Does it match your expectation?
4. Now build the BV oracle with the same $s$ and confirm recovery.

In [ ]:
# Exercise 4: your solution here
n = 3

# Oracle for f(x) = x_0 XOR x_1
oracle_ex4 = QuantumCircuit(n + 1, name='Oracle')
# TODO: add CNOT gates

# TODO: run DJ and check verdict
# TODO: identify s from the measurement
# TODO: build BV oracle with that s and verify recovery

## Summary

In this lesson you:

- Verified phase kickback: setting the ancilla to $|{-}\rangle$ converts the bit oracle $U_f|x\rangle|y\rangle = |x\rangle|y \oplus f(x)\rangle$ into a phase oracle $U_f|x\rangle|{-}\rangle = (-1)^{f(x)}|x\rangle|{-}\rangle$.
- Built Deutsch-Jozsa for $n=3$ and confirmed that constant oracles always produce `'000'` while balanced oracles produce nonzero bitstrings, using one oracle call against the classical worst case of 5.
- Showed that the same circuit solves Bernstein-Vazirani: the measured bitstring equals the hidden secret $s$ with certainty, recovering an $n$-bit string in 1 query against $n$ classical queries.
- Verified the advantage with a classical BV solver and visualised the constructive and destructive interference that make both algorithms work.

**Lesson 8** extends these ideas to an unstructured search problem. Grover's algorithm finds a marked item among $N$ candidates in $O(\sqrt{N})$ oracle calls using amplitude amplification, a more elaborate interference mechanism that applies the oracle multiple times.